In [3]:
import joblib
import shap

In [4]:
shap_value_paths = {
    "longform_dpo_qa_llama3":
        {
            "openai": "PUPPET/experimentalshap_analysis/results/shap_longform_qa_dpo_llama3_2026_0219_191154_openai.joblib",
        },
    "longform_vanila_qa_llama3":
        {
            "openai": "PUPPET/experimentalshap_analysis/results/shap_longform_qa_vanila_llama3_2026_0219_191136_openai.joblib",
        }
}

In [5]:
shap_results = {
    setting_name: {
        model_fn: joblib.load(shap_value_path)
        for model_fn, shap_value_path in shap_value_paths[setting_name].items()
    }
    for setting_name in shap_value_paths.keys()
}

## ROUGE

In [21]:
import re
import html
from typing import (
    List,
    Tuple,
    Dict,
    Any,
    Union,
    Optional,
    Literal,
)

import numpy as np
import pandas as pd
from rouge import Rouge
from tqdm import tqdm
from IPython.display import HTML, display
from weasyprint import HTML as WHTML
from weasyprint import CSS


# ============================================================
# ROUGE-L + LCS（文字ベース）
# ============================================================

def tokenize_with_char_spans(text: str) -> Tuple[List[str], List[Tuple[int, int]]]:
    """
    ROUGE の実装に合わせて空白で区切りつつ、各トークンの (start, end) 文字インデックスを返す。
    """
    tokens: List[str] = []
    spans: List[Tuple[int, int]] = []
    for m in re.finditer(r"\S+", text):
        start, end = m.span()
        tokens.append(text[start:end])
        spans.append((start, end))
    return tokens, spans


def lcs_char_spans_for_pair(pred: str, gt: str) -> List[Tuple[int, int]]:
    """
    pred / gt をトークン化して LCS をとり、
    pred 側の LCS 部分を文字スパンのリスト [(start,end), ...] で返す。
    """
    pred_tokens, pred_spans = tokenize_with_char_spans(pred)
    gt_tokens, _ = tokenize_with_char_spans(gt)

    # ROUGE の実装に揃えるため、小文字化して比較
    a = [t.lower() for t in pred_tokens]
    b = [t.lower() for t in gt_tokens]

    n, m = len(a), len(b)
    dp = [[0] * (m + 1) for _ in range(n + 1)]
    for i in range(n):
        for j in range(m):
            if a[i] == b[j]:
                dp[i + 1][j + 1] = dp[i][j] + 1
            else:
                dp[i + 1][j + 1] = max(dp[i][j + 1], dp[i + 1][j])

    # backtrack -> pred
    i, j = n, m
    idxs: List[int] = []
    while i > 0 and j > 0:
        if a[i - 1] == b[j - 1]:
            idxs.append(i - 1)
            i -= 1
            j -= 1
        elif dp[i - 1][j] >= dp[i][j - 1]:
            i -= 1
        else:
            j -= 1
    idxs.reverse()

    # 連続トークンをまとめて文字スパンに
    spans: List[Tuple[int, int]] = []
    if not idxs:
        return spans

    cur_start, cur_end = pred_spans[idxs[0]]
    prev_idx = idxs[0]
    for k in idxs[1:]:
        tok_start, tok_end = pred_spans[k]
        if k == prev_idx + 1 and tok_start <= cur_end + 1:
            # 連続トークンなら同じスパンにマージ
            cur_end = tok_end
        else:
            spans.append((cur_start, cur_end))
            cur_start, cur_end = tok_start, tok_end
        prev_idx = k
    spans.append((cur_start, cur_end))

    return spans


def rouge_score_waterbench_with_lcs(
    predictions: Union[List[str], List[List[str]]],
    answers: List[List[str]],
) -> Tuple[Union[List[float], List[List[float]]],
           Union[List[List[List[Tuple[int, int]]]], List[List[Tuple[int, int]]]]]:
    """
    もとの rouge_score_waterbench と同じように
    「各 pred について、全 GT の中で Rouge-L が最大のもの」を選びつつ、
    そのベストな pred–GT ペアに対する LCS の文字スパン（予測文側）も一緒に返す。

    戻り値:
      all_scores:  もとの関数と同じ構造の ROUGE-L F スコア
      all_lcs_spans:
          predictions が list[str] のとき:
              List[List[(start,end)]]
              -> [sample_idx][span_idx]
          predictions が list[list[str]] のとき:
              List[List[List[(start,end)]]]
              -> [sample_idx][pred_idx][span_idx]
    """
    rouge = Rouge()

    def _safe_get_rouge_f(pred: str, gt: str) -> float:
        try:
            scores = rouge.get_scores([pred], [gt], avg=True)
        except Exception:
            return 0.0
        return float(scores["rouge-l"]["f"])

    is_list = isinstance(predictions[0], list)

    all_scores = []
    all_lcs_spans = []

    iterator = zip(predictions, answers)
    iterator = tqdm(
        iterator,
        total=len(predictions),
        mininterval=2.0,
        desc="Calculating Rouge-L scores + LCS",
    )

    for pred_or_list, gts in iterator:
        pred_list = pred_or_list if is_list else [pred_or_list]

        sample_scores: List[float] = []
        sample_lcs_spans: List[List[Tuple[int, int]]] = []

        for pred in pred_list:
            best_score = 0.0
            best_idx = 0

            for i, gt in enumerate(gts):
                s = _safe_get_rouge_f(pred, gt)
                if s > best_score:
                    best_score = s
                    best_idx = i

            best_gt = gts[best_idx]
            spans = lcs_char_spans_for_pair(pred, best_gt)

            sample_scores.append(best_score)
            sample_lcs_spans.append(spans)

        all_scores.append(sample_scores if is_list else sample_scores[0])
        all_lcs_spans.append(sample_lcs_spans if is_list else sample_lcs_spans[0])

    return all_scores, all_lcs_spans


# ============================================================
# 文字レベルの汎用 span ヘルパ
# ============================================================

def spans_to_mask(length: int, spans: List[Tuple[int, int]]) -> np.ndarray:
    """
    長さ length の bool 配列を作り、spans (start,end) に True を立てる。
    """
    mask = np.zeros(length, dtype=bool)
    for start, end in spans:
        start = max(0, min(length, start))
        end   = max(0, min(length, end))
        if end > start:
            mask[start:end] = True
    return mask


def mask_to_spans(mask: np.ndarray) -> List[Tuple[int, int]]:
    """
    bool 配列から [start, end) のスパン列を復元。
    """
    spans: List[Tuple[int, int]] = []
    start = None
    for i, flag in enumerate(mask):
        if flag and start is None:
            start = i
        elif not flag and start is not None:
            spans.append((start, i))
            start = None
    if start is not None:
        spans.append((start, len(mask)))
    return spans


def span_stats_from_mask(mask: np.ndarray) -> Dict[str, float]:
    """
    文字レベルの bool mask から簡単な統計量を返す。
      - num_spans: True が連続している区間の数
      - avg_span_len: その平均長さ（文字数）
      - coverage: True の割合（有効文字数 / 全体の文字数）
    """
    L = len(mask)
    spans = mask_to_spans(mask)
    strong_chars = int(mask.sum())
    num_spans = len(spans)
    avg_len = float(np.mean([end - start for start, end in spans])) if spans else 0.0
    coverage = strong_chars / L if L > 0 else 0.0
    return {
        "num_spans": num_spans,
        "avg_span_len": avg_len,
        "coverage": coverage,
    }


def span_stats_from_spans(text_len: int, spans: List[Tuple[int, int]]) -> Dict[str, float]:
    """
    文字長と spans から stats を計算する関数。
    """
    mask = spans_to_mask(text_len, spans)
    return span_stats_from_mask(mask)


# ============================================================
# Detector SHAP: token ベースのスコア & span stats
# ============================================================

def token_mask_to_spans(mask: np.ndarray) -> List[Tuple[int, int]]:
    """
    bool 配列から [start, end) のトークンスパンを復元。
    例) mask=[F,T,T,F,T] → [(1,3),(4,5)]
    """
    spans: List[Tuple[int, int]] = []
    start = None
    for i, flag in enumerate(mask):
        if flag and start is None:
            start = i
        elif not flag and start is not None:
            spans.append((start, i))
            start = None
    if start is not None:
        spans.append((start, len(mask)))
    return spans


def token_span_stats_from_mask(mask: np.ndarray) -> Dict[str, float]:
    """
    トークンレベルの bool mask から簡単な統計量を返す。
      - num_spans: True が連続している区間の数
      - avg_span_len: その平均長さ（トークン数）
      - coverage: True の割合（有効トークン数 / 全トークン数）
    """
    L = len(mask)
    spans = token_mask_to_spans(mask)
    strong_tokens = int(mask.sum())
    num_spans = len(spans)
    avg_len = float(np.mean([end - start for start, end in spans])) if spans else 0.0
    coverage = strong_tokens / L if L > 0 else 0.0
    return {
        "num_spans": num_spans,
        "avg_span_len": avg_len,
        "coverage": coverage,
    }


def extract_token_scores_for_class(
    shap_exp: Any,
    class_index: int = 0,
    mode: Literal["pos", "neg", "abs", "signed"] = "pos",
) -> np.ndarray:
    """
    shap_exp.values: (L, C) または (1, L, C) を想定。
    指定クラス class_index への寄与を取り出し、mode に応じてスコア化する。

    戻り値: shape (L,)
    """
    values = np.array(shap_exp.values)
    if values.ndim == 3:
        # (1, L, C) → (L, C) に潰す
        values = values[0]
    v = values[:, class_index]  # class_index への寄与 (L,)

    if mode == "pos":
        return np.maximum(v, 0.0)      # class_index 方向へ押し上げる部分
    elif mode == "neg":
        return np.maximum(-v, 0.0)     # class_index から遠ざける部分
    elif mode == "abs":
        return np.abs(v)               # 強さだけ見る
    elif mode == "signed":
        return v                       # 符号付きそのまま
    else:
        raise ValueError(f"unknown mode: {mode}")


def det_shap_token_span_stats(
    shap_det: Any,
    mode: Literal["pos", "neg", "abs", "signed"] = "pos",
    shap_quantile: float = 0.99,
    shap_threshold: float = 0.99,
    global_cmax: float | None = None,
    class_index: int = 0,
) -> Dict[str, Any]:
    """
    Detector 用 SHAP (shap_det) について、
    「SHAP の tokenizer の token 単位」で span stats を計算する。

    正規化:
      - まずトークンスコアを mode に応じて計算
      - その絶対値の上位 shap_quantile を cmax とし 0〜1 にスケーリング
        （global_cmax が与えられていればそれを優先）
      - スケーリング後のスコアが shap_threshold 以上のトークンを「有効」とみなす
    """
    # 1. トークンスコアを取得
    scores = extract_token_scores_for_class(
        shap_det,
        class_index=class_index,
        mode=mode,
    )  # (L,)

    v_abs = np.abs(scores)

    # 2. cmax を決める（global_cmax 優先）
    if global_cmax is not None:
        cmax = global_cmax
    else:
        if shap_quantile is not None:
            cmax = float(np.quantile(v_abs, shap_quantile)) if v_abs.size > 0 else 0.0
        else:
            cmax = float(v_abs.max()) if v_abs.size > 0 else 0.0

    # 3. 0〜1 正規化
    if cmax <= 0:
        scores_norm = np.zeros_like(scores)
    else:
        scores_norm = np.clip(v_abs, 0, cmax) / cmax  # (L,)

    # 4. 有効トークンの mask
    mask = scores_norm >= shap_threshold

    # 5. トークン列
    data = shap_det.data
    if isinstance(data, tuple):
        tokens = list(map(str, data[0]))
    else:
        tokens = list(map(str, data))
    assert len(tokens) == scores.shape[0], "tokens と scores の長さが一致しません"

    spans = token_mask_to_spans(mask)
    stats = token_span_stats_from_mask(mask)

    return {
        "tokens": tokens,
        "scores_raw": scores,
        "scores_norm": scores_norm,
        "mask": mask,
        "spans": spans,
        "stats": stats,
    }


# ============================================================
# Detector × ROUGE-L 重ね描画（tokenで判定→文字を塗る）
# ============================================================

def overlay_det_shap_and_rouge_lcs(
    shap_det,
    pred_text: str,
    lcs_char_spans: List[Tuple[int, int]],
    mode: Literal["pos", "neg", "abs", "signed"] = "pos",
    shap_quantile: float = 0.99,
    shap_threshold: float = 0.7,
    global_cmax: float | None = None,
    det_span_color: str = "red",
    lcs_span_color: str = "blue",
    both_span_color: str = "purple",
    output_background_color: str = "#ffffff",
    output_text_color: str = "#000000",
    return_html_str: bool = False,
) -> HTML | str:
    """
    Detector SHAP (label 0) と ROUGE-L の LCS を 1枚の HTML として重ねて表示。

    - Detector 側は token-wise に「強い token」を決めてから、
      その token に含まれる文字を赤/紫で着色する。
    - ROUGE 側は文字スパン (lcs_char_spans) そのまま。

    mode:
      "pos"   : class 0 方向に押し上げる寄与のみ (max(v,0))
      "neg"   : class 0 から遠ざける寄与のみ (max(-v,0))
      "abs"   : |v|
      "signed": v そのまま（強さ判定は |v| ベース）
    """

    # --- 1. SHAP の token とテキストを復元 ---
    data = shap_det.data
    if isinstance(data, tuple):
        tokens = [str(t) for t in data[0]]
    else:
        tokens = [str(t) for t in data]

    text_from_shap = "".join(tokens)
    assert text_from_shap == pred_text, (
        "Text from SHAP and pred_text must be the same.\n"
        f"SHAP: {text_from_shap}\n"
        f"PRED: {pred_text}"
    )

    N = len(pred_text)

    # --- 2. token-wise のスコアを取る (class 0) ---
    scores = extract_token_scores_for_class(
        shap_det, class_index=0, mode=mode
    )   # (L,)

    # 「強さ」として使う値 base_for_norm を mode に応じて決める
    if mode in ("pos", "neg", "abs"):
        base_for_norm = scores          # すでに >= 0
    else:  # mode == "signed"
        base_for_norm = np.abs(scores)

    # --- 3. token-wise に 0〜1 正規化 (global_cmax 優先) ---
    if global_cmax is not None:
        cmax = global_cmax
    else:
        if shap_quantile is not None:
            cmax = float(np.quantile(base_for_norm, shap_quantile)) if base_for_norm.size > 0 else 0.0
        else:
            cmax = float(base_for_norm.max()) if base_for_norm.size > 0 else 0.0

    if cmax <= 0:
        scores_norm = np.zeros_like(scores)
    else:
        scores_norm = np.clip(base_for_norm, 0, cmax) / cmax    # (L,) in [0,1]

    # --- 4. token-wise に「強いかどうか」のマスクを作る ---
    token_mask = scores_norm >= shap_threshold    # (L,) bool

    # --- 5. token_mask を文字マスクに展開 ---
    shap_mask = np.zeros(N, dtype=bool)
    offset = 0
    for tok, is_strong in zip(tokens, token_mask):
        L_tok = len(tok)
        if L_tok > 0 and is_strong:
            shap_mask[offset:offset + L_tok] = True
        offset += L_tok

    # --- 6. ROUGE-L (LCS) 側の文字マスク ---
    lcs_mask = spans_to_mask(N, lcs_char_spans)

    # --- 7. 色重ねの HTML を作る ---
    color_map = {
        "orange": "rgba(255, 140, 0, 0.7)",
        "blue":   "rgba(58, 171, 210, 0.7)",
        "brown":  "rgba(150, 80, 0, 0.9)",
        "red":    "rgba(223, 86, 86, 0.7)",
        "green":  "rgba(0, 255, 0, 0.7)",
        "purple": "rgba(160, 110, 190, 0.7)",
    }

    spans_html: List[str] = []
    for i, ch in enumerate(pred_text):
        s = shap_mask[i]
        l = lcs_mask[i]

        if s and l:
            bg = color_map[both_span_color]   # 紫
        elif s:
            bg = color_map[det_span_color]    # 赤
        elif l:
            bg = color_map[lcs_span_color]    # 青
        else:
            bg = "rgba(0, 0, 0, 0.0)"         # 透明

        esc_ch = html.escape(ch)
        spans_html.append(
            f"<span style='background-color:{bg}'>{esc_ch}</span>"
        )

    html_body = "".join(spans_html)

    # 改行なしで余白を増やさない
    html_full = (
        f"<div style=\"font-family: 'Courier New', monospace; "
        f"line-height: 1.6; "
        f"white-space: pre-wrap; "
        f"background-color: {output_background_color}; "
        f"color: {output_text_color};\">"
        f"{html_body}"
        f"</div>"
    )

    if return_html_str:
        return html_full
    else:
        return HTML(html_full)


# ============================================================
# 可視化用（入力テキストを同じスタイルで描画）
# ============================================================

def render_input_text_overlay_style(
    input_text: str,
    background: str = "#ffffff",
    text_color: str = "#000000",
    return_html_str: bool = False,
):
    """
    モデルへの入力テキストを、
    Detector×ROUGE-L 重ね描画と同じスタイルで表示するための関数。

    - フォント / 行間 / 改行の扱い / 背景色 を重ね描画と揃える
    - 文字には着色しない（プレーンなテキスト）
    """
    body = html.escape(input_text)

    html_full = (
        f"<div style=\"font-family: 'Courier New', monospace; "
        f"line-height: 1.6; "
        f"white-space: pre-wrap; "
        f"background-color: {background}; "
        f"color: {text_color};\">"
        f"{body}"
        f"</div>"
    )

    if return_html_str:
        return html_full
    else:
        return HTML(html_full)


# ============================================================
# HTML style の後処理・ファイル変換
# ============================================================

def parse_style(style_str: str) -> Dict[str, str]:
    props: Dict[str, str] = {}
    for part in style_str.split(";"):
        part = part.strip()
        if not part or ":" not in part:
            continue
        k, v = part.split(":", 1)
        props[k.strip().lower()] = v.strip()
    return props


def build_style(props: Dict[str, str]) -> str:
    return "; ".join(f"{k}: {v}" for k, v in props.items())


def tweak_overlay_html_style(
    html_str: str,
    line_height: Optional[str] = None,
    letter_spacing: Optional[str] = None,
    word_spacing: Optional[str] = None,
    font_size: Optional[str] = None,
    max_width: Optional[str] = None,
    padding: Optional[str] = None,
    strip_div_inner_whitespace: bool = False,
) -> str:
    """
    overlay 用 HTML に対して style の post-edit と、
    （オプションで）<div> 内側の先頭/末尾の改行・空白を削る。
    """

    # style="" を見つけてプロパティを書き換える
    m_style = re.search(r'<div\s+[^>]*style="([^"]*)"', html_str)
    if m_style:
        old_style = m_style.group(1)
        props = parse_style(old_style)

        if line_height is not None:
            props["line-height"] = line_height
        if letter_spacing is not None:
            props["letter-spacing"] = letter_spacing
        if word_spacing is not None:
            props["word-spacing"] = word_spacing
        if font_size is not None:
            props["font-size"] = font_size
        if max_width is not None:
            props["max-width"] = max_width
        if padding is not None:
            props["padding"] = padding

        new_style = build_style(props)

        start, end = m_style.span(1)  # style="ここ"
        html_str = html_str[:start] + new_style + html_str[end:]

    html_str = html_str.strip()

    if strip_div_inner_whitespace:
        m_div = re.search(
            r'(<div\s+[^>]*>)(.*?)(</div>)',
            html_str,
            flags=re.DOTALL,
        )
        if m_div:
            open_tag = m_div.group(1)
            inner    = m_div.group(2)
            close_tag= m_div.group(3)
            trimmed_inner = inner.strip(" \t\r\n")
            new_div = open_tag + trimmed_inner + close_tag
            start, end = m_div.span()
            html_str = html_str[:start] + new_div + html_str[end:]

    return html_str


def tweak_overlay_html_file(
    input_path: str,
    output_path: str,
    line_height: Optional[str] = None,
    letter_spacing: Optional[str] = None,
    word_spacing: Optional[str] = None,
    font_size: Optional[str] = None,
    max_width: Optional[str] = None,
    padding: Optional[str] = None,
):
    """
    HTMLファイルを読み込んで tweak_overlay_html_style を適用し、
    別ファイルとして保存するためのラッパ。
    """
    with open(input_path, "r", encoding="utf-8") as f:
        html_str = f.read()

    new_html = tweak_overlay_html_style(
        html_str,
        line_height=line_height,
        letter_spacing=letter_spacing,
        word_spacing=word_spacing,
        font_size=font_size,
        max_width=max_width,
        padding=padding,
    )

    with open(output_path, "w", encoding="utf-8") as f:
        f.write(new_html)


def get_css_and_doc(inner_html: str, width_px: int = 900, height_px: int = -1):
    css = CSS(string=f"""
        @page {{
            size: {width_px}px {height_px}px;
            margin: 0;
        }}
        html, body {{
            margin: 0;
            padding: 0;
        }}
    """)
    doc = f"<html><body>{inner_html}</body></html>"
    return css, doc


def save_html_as_pdf(
    inner_html: str,
    output_path: str,
    width_px: int = 900,
    height_px: int = -1,
):
    css, doc = get_css_and_doc(inner_html, width_px, height_px)
    WHTML(string=doc).write_pdf(output_path, stylesheets=[css])
    print(f"Saved 1-page figure PDF to {output_path}")


def save_html_as_png(
    inner_html: str,
    output_path: str,
    width_px: int = 900,
    height_px: int = -1,
):
    css, doc = get_css_and_doc(inner_html, width_px, height_px)
    WHTML(string=doc).write_png(output_path, stylesheets=[css])
    print(f"Saved 1-page figure PNG to {output_path}")


# ============================================================
# 統計まとめ (ROUGE + Detector) ＆ サンプル表示
# ============================================================

def get_stats_df(
    predictions,
    lcs_spans,
    shap_det,
    mode: str = "pos",
    shap_quantile: float = 0.99,
    shap_threshold: float = 0.99,
    pairwise_cmax: Optional[List[List[float]]] = None,
    global_cmax: float | None = None,
) -> pd.DataFrame:
    """
    shap_before[i][j], shap_after[i][j] という 2 次元構造を想定し、
    各ペア (i, j) ごとに

        v_before = SHAP(before, class 0, mode)
        v_after  = SHAP(after,  class 0, mode)

    の絶対値を併合した分布の上位 quantile を cmax_ij として求める。

    戻り値:
      pairwise_cmax[i][j] = cmax_ij  (float)
    """
    stats = []
    for i, (pred, span, det) in enumerate(zip(predictions, lcs_spans, shap_det)):
        sub_stats_list = []
        for j, (p, s, d) in enumerate(zip(pred, span, det)):
            st_rouge = span_stats_from_spans(len(p), s)

            # --- cmax_ij を決める ---
            if pairwise_cmax is not None:
                cmax_ij = pairwise_cmax[i][j]
            else:
                cmax_ij = global_cmax

            det_res = det_shap_token_span_stats(
                d,
                mode=mode,
                shap_quantile=shap_quantile,
                shap_threshold=shap_threshold,
                global_cmax=cmax_ij,
                class_index=0,
            )
            st_det = det_res["stats"]

            sub_stats_list.append(
                {
                    "pred_len": len(p),
                    "rouge_num_spans": st_rouge["num_spans"],
                    "rouge_avg_span_len": st_rouge["avg_span_len"],
                    "rouge_coverage": st_rouge["coverage"],
                    "det_num_spans": st_det["num_spans"],
                    "det_avg_span_len": st_det["avg_span_len"],
                    "det_coverage": st_det["coverage"],
                }
            )
        stats.append(
            {
                f"generation_{k}": sub_stats
                for k, sub_stats in enumerate(sub_stats_list)
            }
        )
    stats_df = pd.DataFrame(stats)
    return stats_df


def compute_pairwise_cmax_for_models(
    shap_before: List[List[Any]],
    shap_after: List[List[Any]],
    mode: str = "pos",
    quantile: float = 0.99,
) -> List[List[float]]:
    """
    predictions: List[List[str]]    各サンプルごとに generation のリスト
    lcs_spans:   同じ構造で、各 prediction の LCS スパン（文字indexベース）
    shap_det:    同じ構造で、各 prediction の Detector 用 SHAP

    Detector 側は token ベースの stats（det_shap_token_span_stats）を使用。

    スケーリング:
      - pairwise_cmax が与えられていれば、(i,j) ごとの cmax_ij を使用
      - そうでなければ global_cmax を使用
      - どちらも None のときはサンプル内正規化（det_shap_token_span_stats 側）
    """
    pairwise_cmax: List[List[float]] = []
    for row_before, row_after in zip(shap_before, shap_after):
        row_cmax: List[float] = []
        for sd_before, sd_after in zip(row_before, row_after):
            v_b = extract_token_scores_for_class(
                sd_before, class_index=0, mode=mode
            ).reshape(-1)
            v_a = extract_token_scores_for_class(
                sd_after, class_index=0, mode=mode
            ).reshape(-1)
            all_scores = np.concatenate(
                [np.abs(v_b), np.abs(v_a)]
            ) if (v_b.size + v_a.size) > 0 else np.array([], dtype=float)
            if all_scores.size == 0:
                cmax_ij = 0.0
            else:
                cmax_ij = float(np.quantile(all_scores, quantile))
            row_cmax.append(cmax_ij)
        pairwise_cmax.append(row_cmax)
    return pairwise_cmax


def show_sample(
    i: int,
    j: int,
    predictions,
    lcs_spans_all,
    shap_det,
    mode: str = "pos",
    shap_quantile: float = 0.99,
    shap_threshold: float = 0.5,
    pairwise_cmax: Optional[List[List[float]]] = None,
    global_cmax: float | None = None,
    return_html_str: bool = False,
):
    """
    1サンプル・1生成分を簡単に眺めるためのユーティリティ。
    stats は:
      - ROUGE: 文字ベース
      - Detector: token ベース

    スケーリング:
      - pairwise_cmax があれば pairwise_cmax[i][j] を使用
      - なければ global_cmax を使用
      - どちらも None のときはサンプル内正規化
    """
    pred_text = predictions[i][j] if isinstance(predictions[0], list) else predictions[i]
    lcs_spans = lcs_spans_all[i][j] if isinstance(predictions[0], list) else lcs_spans_all[i]
    shap_det_ij = shap_det[i][j]

    # どの cmax を使うか決める
    if pairwise_cmax is not None:
        cmax_ij = pairwise_cmax[i][j]
    else:
        cmax_ij = global_cmax

    # ROUGE 側 stats
    stats_rouge = span_stats_from_spans(len(pred_text), lcs_spans)

    # Detector 側 stats（token ベース）
    det_res = det_shap_token_span_stats(
        shap_det_ij,
        mode=mode,
        shap_quantile=shap_quantile,
        shap_threshold=shap_threshold,
        global_cmax=cmax_ij,
        class_index=0,
    )
    stats_det = det_res["stats"]

    if not return_html_str:
        print(f"stats_rouge: {stats_rouge}, stats_det: {stats_det}")

    html = overlay_det_shap_and_rouge_lcs(
        shap_det_ij,
        pred_text,
        lcs_spans,
        mode=mode,
        shap_quantile=shap_quantile,
        shap_threshold=shap_threshold,
        global_cmax=cmax_ij,
        return_html_str=return_html_str,
    )

    if return_html_str:
        return html
    else:
        display(html)

In [12]:
import pandas as pd

df_before = pd.read_json(
    "PUPPET/experimentalvllm_gen/data/baselines/vanila/generation_vllm_longform_qa_llama3_dpo_baseline_2025_1204_141800.jsonl",
    lines=True,
)
df_after = pd.read_json(
    "PUPPET/experimentalvllm_gen/data/ours/generation_vllm_longform_qa_llama3_dpo_lrx100_drstd_5000_2026_0206_133612.jsonl",
    lines=True,
)

predictions_before = df_before['vllm_model_output'].tolist()
predictions_after = df_after['vllm_model_output'].tolist()
answers_before = df_before['outputs'].tolist()
answers_after = df_after['outputs'].tolist()

In [13]:
_, lcs_spans_before = rouge_score_waterbench_with_lcs(predictions_before, answers_before)
_, lcs_spans_after = rouge_score_waterbench_with_lcs(predictions_after, answers_after)

In [14]:
shap_det_before = shap_results['longform_vanila_qa_llama3']['openai']
shap_det_after = shap_results['longform_dpo_qa_llama3']['openai']

In [15]:
mode = "pos"
quantile = 0.99
shap_threshold = 0.4
pairwise_cmax = compute_pairwise_cmax_for_models(shap_det_before, shap_det_after, mode=mode, quantile=quantile)

span_config = {
    "mode": mode,
    "shap_quantile": quantile,
    "shap_threshold": shap_threshold,
    "pairwise_cmax": pairwise_cmax,
}

In [16]:
stats_df_before = get_stats_df(predictions_before, lcs_spans_before, shap_det_before, **span_config)
stats_df_after = get_stats_df(predictions_after, lcs_spans_after, shap_det_after, **span_config)

In [17]:
stats_df_before_flatten = pd.json_normalize(stats_df_before.stack().reset_index(drop=True))
stats_df_after_flatten = pd.json_normalize(stats_df_after.stack().reset_index(drop=True))
stats_df_gap = stats_df_after_flatten-stats_df_before_flatten

In [18]:
stats_df_gap.describe()

In [ ]:
import datetime

stats_out_dir = "PUPPET/experimentalshap_analysis/stats"

current_time = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
output_config = {
    "index": False,
    "orient": "records",
    "lines": True,
    "force_ascii": False,
} 

stats_df_gap.to_json(f"{stats_out_dir}/stats_df_gap_{current_time}.jsonl", **output_config)
stats_df_before_flatten.to_json(f"{stats_out_dir}/stats_df_before_flatten_{current_time}.jsonl", **output_config)
stats_df_after_flatten.to_json(f"{stats_out_dir}/stats_df_after_flatten_{current_time}.jsonl", **output_config)

In [58]:
sample_idx = 129
pred_idx = 1

show_sample(sample_idx, pred_idx, predictions_before, lcs_spans_before, shap_det_before, **span_config)
print("\n" + "↓"*1000 + "\n")
show_sample(sample_idx, pred_idx, predictions_after, lcs_spans_after, shap_det_after, **span_config)

In [59]:
html_bef = show_sample(sample_idx, pred_idx, predictions_before, lcs_spans_before, shap_det_before, return_html_str=True, **span_config)
html_aft = show_sample(sample_idx, pred_idx, predictions_after, lcs_spans_after, shap_det_after, return_html_str=True, **span_config)

In [60]:
# 入力文
input_text = df_before['input'].tolist()[sample_idx]
html_input = render_input_text_overlay_style(
    input_text,
    background="#ffffff",
    text_color="#000000",
    return_html_str=True,
)

In [61]:
tewak_config_input = {
    "line_height": "1.0",
    "letter_spacing": "0.02em",
    "font_size": "20pt",
    "max_width": "2000px",
    "padding": "0px 0px",
    "strip_div_inner_whitespace": True,
}


html_input_tweaked = tweak_overlay_html_style(
    html_input,
    **tewak_config_input,
)
display(HTML(html_input_tweaked))


tewak_config = {
    "line_height": "1.0",
    "letter_spacing": "0.02em",
    "font_size": "20pt",
    "max_width": "900px",
    "padding": "0px 0px",
    "strip_div_inner_whitespace": True,
}

html_bef_tweaked = tweak_overlay_html_style(
    html_bef,
    **tewak_config,
)
display(HTML(html_bef_tweaked))

html_aft_tweaked = tweak_overlay_html_style(
    html_aft,
    **tewak_config,
)
display(HTML(html_aft_tweaked))


In [ ]:
import os
import datetime

html_save_dir = "PUPPET/experimentalshap_analysis/figures"
current_time = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')

run_name = "eli_5_paper"

save_config = {
    "before": {
        "name": f"{run_name}_before_{current_time}",
        "html": html_bef_tweaked,
        "pdf_config": {"height_px": 940},
        "png_config": {"height_px": 940},
    },
    "after": {
        "name": f"{run_name}_after_{current_time}",
        "html": html_aft_tweaked,
        "pdf_config": {"height_px": 860},
        "png_config": {"height_px": 860},
    },
    "input": {
        "name": f"{run_name}_input_{current_time}",
        "html": html_input_tweaked,
        "pdf_config": {"height_px": 110, "width_px": 1800},
        "png_config": {"height_px": 110, "width_px": 1800},
    },
}

for key, config in save_config.items():
    save_html_as_png(config["html"], f"{html_save_dir}/{config['name']}.png", **config["png_config"])
    save_html_as_pdf(config["html"], f"{html_save_dir}/{config['name']}.pdf", **config["pdf_config"])